# IS insertion ~ essential genes
Previously (ipynb: 02) I analyzed the significance of the correlation
between the number of essential genes and the number of IS insertions.
Here, according to the reviewer, I will analyze

IS insertion ~ pre-existing IS + essential genes + round

The previous analysis used the data of mapping onto MDS42 genome.
The distribution of IS is meaningless because there was only one IS in the strain.

Here, I compare either with the previous sequencing round, or with the FACS generation.

Same as R01 but with 10kbp windows.

## Comparison with the FACS generated ancestor
- count the number of essential genes in each window 
- count the number of IS insertions in each window

In [1]:
import Pkg
using RCall
using Random
using DataFrames, DataFramesMeta, Chain
using Revise
using CSV
using StatsBase
using Statistics
using Arrow

R"""
Sys.setlocale("LC_COLLATE","C") # order by dictionary order
library(tidyverse)
library(cowplot)
library(latex2exp)
library(stats)
library(scales)
library(lemon)
library(gghalves)
library(ggbeeswarm)
library(ggpointdensity)
library(viridis)
library(forcats)
library(MASS)
library(sfsmisc)
library(ggnewscale)
library(ggrastr)
library(car)
library(arrow)
library(performance)
library(pscl)
cbp <- c("#999999", "#E69F00", "#56B4E9", "#009E73", 
                       "#F0E442", "#0072B2", "#D55E00", "#CC79A7")
select <- dplyr::select
filter <- dplyr::filter
lag <- dplyr::lag
""";

┌ Warning: RCall.jl: ── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
│ ✔ dplyr     1.1.3     ✔ readr     2.1.4
│ ✔ forcats   1.0.0     ✔ stringr   1.5.0
│ ✔ ggplot2   3.4.3     ✔ tibble    3.2.1
│ ✔ lubridate 1.9.2     ✔ tidyr     1.3.0
│ ✔ purrr     1.0.2     
│ ── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
│ ✖ dplyr::filter() masks stats::filter()
│ ✖ dplyr::lag()    masks stats::lag()
│ ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172
┌ Warning: RCall.jl: 
│ Attaching package: ‘cowplot’
│ 
│ The following object is masked from ‘package:lubridate’:
│ 
│     stamp
│ 
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172
┌ Warning: RCall.jl: 
│ Attaching package: ‘scales’
│ 
│ The following object is masked from ‘package:purrr’:
│ 
│     discard
│ 
│ The following object is masked from ‘package:readr’:
│ 
│     co

In [2]:
R"""
base_dir = file.path("exp", "multiple_runs14")
exp_dir = file.path(base_dir, "export", "classify_IS_events")
dat_dir = file.path(base_dir, "data", "ins_del_analysis")
fig_dir = file.path(base_dir, "figs", "R01_INS_ESS_corr")
tmp_dir = file.path(base_dir, "tmp", "R01_INS_ESS_corr")
file_name_base = "20250204_"

dir.create(fig_dir, showWarnings = FALSE, recursive = TRUE)
dir.create(tmp_dir, showWarnings = FALSE, recursive = TRUE)

ess_pos_df <- read_csv(file.path(dat_dir, paste0(file_name_base, "essential_gene_pos.csv")))
ess_pos_df %>% head(2)
"""

┌ Warning: RCall.jl: Rows: 40055 Columns: 15
│ ── Column specification ────────────────────────────────────────────────────────
│ Delimiter: ","
│ chr  (2): gene, line
│ dbl (12): pident, length, mismatch, gapopen, qstart, qend, sstart, send, eva...
│ lgl  (1): duplicated
│ 
│ ℹ Use `spec()` to retrieve the full column specification for this data.
│ ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


RObject{VecSxp}
# A tibble: 2 × 15
  gene  pident length mismatch gapopen qstart  qend  sstart    send evalue
  <chr>  <dbl>  <dbl>    <dbl>   <dbl>  <dbl> <dbl>   <dbl>   <dbl>  <dbl>
1 mukB    100    4460        0       0      1  4460 1216300 1211841      0
2 rpoC    100.   4223        2       0      1  4223  259602  263824      0
# ℹ 5 more variables: bitscore <dbl>, qlen <dbl>, line <chr>, gen_id <dbl>,
#   duplicated <lgl>


In [3]:
R"""
is_pos_df <- read_csv(file.path(dat_dir, paste0(file_name_base, "is_position_in_genome.csv")),comment = "#")
is_pos_df %>% head(2)
"""

┌ Warning: RCall.jl: Rows: 2385 Columns: 7
│ ── Column specification ────────────────────────────────────────────────────────
│ Delimiter: ","
│ chr (2): IS_strand, Line
│ dbl (5): start, end, length, IS_id, Gen
│ 
│ ℹ Use `spec()` to retrieve the full column specification for this data.
│ ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


RObject{VecSxp}
# A tibble: 2 × 7
   start    end IS_strand length IS_id Line    Gen
   <dbl>  <dbl> <chr>      <dbl> <dbl> <chr> <dbl>
1 476928 480020 reverse     3093     0 L01-1     1
2 557167 560259 reverse     3093     1 L01-1     1


In [4]:
R"""
ins_pos_prevgen <- read_csv(file.path(exp_dir, "IS_positions_in_ref_genome.csv"))
ins_pos_prevgen %>% head(2)
"""

┌ Warning: RCall.jl: Rows: 6410 Columns: 12
│ ── Column specification ────────────────────────────────────────────────────────
│ Delimiter: ","
│ chr (5): flank, genome, position_status, sv, Line
│ dbl (5): pos_id, pos, cluster_id, insert_id, Gen
│ lgl (2): IS_strand, flankMatchDir
│ 
│ ℹ Use `spec()` to retrieve the full column specification for this data.
│ ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


RObject{VecSxp}
# A tibble: 2 × 12
  pos_id IS_strand flank    pos genome cluster_id insert_id flankMatchDir
   <dbl> <lgl>     <chr>  <dbl> <chr>       <dbl>     <dbl> <lgl>        
1      0 FALSE     f     476928 Ref            26         1 FALSE        
2      1 FALSE     f     557167 Ref            23         4 FALSE        
# ℹ 4 more variables: position_status <chr>, sv <chr>, Line <chr>, Gen <dbl>


In [5]:
R"""
genome_size_df <- read_csv(file.path(exp_dir, "genome_stats.csv"))
tail(genome_size_df, 2)
"""

┌ Warning: RCall.jl: Rows: 132 Columns: 21
│ ── Column specification ────────────────────────────────────────────────────────
│ Delimiter: ","
│ chr  (8): IS_Detect_ID, gen, file_name, Anc, Prefix, sample_name_raw, Contig...
│ dbl (12): ParentLine, SubLine, RecA, gen_id, genome_size, is_cnt, is_length,...
│ lgl  (1): Complete
│ 
│ ℹ Use `spec()` to retrieve the full column specification for this data.
│ ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


RObject{VecSxp}
# A tibble: 2 × 21
  IS_Detect_ID ParentLine SubLine gen   file_name             Anc    RecA Prefix
  <chr>             <dbl>   <dbl> <chr> <chr>                 <chr> <dbl> <chr> 
1 L11-3_G20            11       3 20    20231003/L11-3_G20.f… r05       0 L11-3 
2 L11-4_G20            11       4 20    20231003/L11-4_G20.f… r05       0 L11-4 
# ℹ 13 more variables: sample_name_raw <chr>, Contig_Date <chr>,
#   Complete <lgl>, gen_id <dbl>, File <chr>, genome_size <dbl>, is_cnt <dbl>,
#   is_length <dbl>, ins_pos_cnt_raw <dbl>, ins_pos_cnt <dbl>, new_cnt <dbl>,
#   lost_end_cnt <dbl>, simple_insertion_cnt <dbl>


In [6]:
using DataFrames

# Function to count overlaps
function within_range!(range_df::DataFrame, pos_df::DataFrame)
    range_df[!, :overlap_count] .= 0  # Initialize column

    for i in 1:nrow(pos_df)
        pos = pos_df[i, :]
        rf, re = range_df.first, range_df.last
        pf, pe = pos.first, pos.last
        range_df[:, :overlap_count] .+= ((rf .< pe) .& (pf .< re)) .|| ((pf .< re) .& (rf .< pe))
    end
    return range_df
end

within_range! (generic function with 1 method)

In [7]:
@rget ess_pos_df is_pos_df ins_pos_prevgen genome_size_df;

In [8]:
ess_pos_df = ess_pos_df[ess_pos_df.duplicated .== false, :] # remove duplicates
ess_pos_df |> x -> first(x, 2)

Row,gene,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,line,gen_id,duplicated
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64,Bool
1,mukB,100.0,4460.0,0.0,0.0,1.0,4460.0,1.2163e6,1.21184e6,0.0,8237.0,4460.0,L01-1,1.0,false
2,rpoC,99.953,4223.0,2.0,0.0,1.0,4223.0,259602.0,263824.0,0.0,7788.0,4223.0,L01-1,1.0,false


In [9]:
# Example values
line = "L01-1"
gen_id = 1
buffer = 20
range_length = 10000
genome_size_ = @chain genome_size_df begin
    @subset(:Prefix .== line, :gen_id .== gen_id)
end
genome_size_ = genome_size_.genome_size[1]

n_range = trunc(Int, genome_size_ / range_length)

df_ = ess_pos_df[ess_pos_df.line .== line .&& ess_pos_df.gen_id .== gen_id, 
                 [:sstart, :send]]

df_[!, :first] = min.(df_.sstart, df_.send) .- buffer
df_[!, :last] = max.(df_.sstart, df_.send) .+ buffer

# Generate range DataFrame
range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
                      last=collect(range_length:range_length:genome_size_))
# int
range_df.first = Int.(range_df.first)
range_df.last = Int.(range_df.last)

# Compute overlaps
within_range!((range_df), df_)
range_df[!, :line] .= line
range_df[!, :gen_id] .= gen_id

# Show first two rows
first(range_df, 2)

Row,first,last,overlap_count,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [10]:
range_df[range_df.overlap_count .> 0, :] |> x -> first(x, 5)

Row,first,last,overlap_count,line,gen_id
,Int64,Int64,Int64,String,Int64
1,20001,30000,1,L01-1,1
2,40001,50000,1,L01-1,1
3,50001,60000,3,L01-1,1
4,60001,70000,2,L01-1,1
5,100001,110000,1,L01-1,1


In [11]:
# sort with ssatrt
sort(df_, :first)[1:5, :]

Row,sstart,send,first,last
,Float64,Float64,Float64,Float64
1,21215.0,21289.0,21195.0,21309.0
2,40675.0,41933.0,40655.0,41953.0
3,56633.0,56708.0,56613.0,56728.0
4,56767.0,56842.0,56747.0,56862.0
5,56992.0,57067.0,56972.0,57087.0


In [12]:
# get the essential gene counts
lines = unique(ess_pos_df.line)
gen_ids = unique(ess_pos_df.gen_id)
buffer = 20
range_length = 10000

range_df_ar = []

for line in lines
	for gen_id in gen_ids
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]
		
		n_range = trunc(Int, genome_size_ / range_length)
		df_ = ess_pos_df[ess_pos_df.line .== line .&& ess_pos_df.gen_id .== gen_id,
		                 [:sstart, :send]]
		df_[!, :first] = min.(df_.sstart, df_.send) .- buffer
		df_[!, :last] = max.(df_.sstart, df_.send) .+ buffer

		# Generate range DataFrame
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)

		# Compute overlaps
		within_range!((range_df), df_)
		rename!(range_df, :overlap_count => :ess_cnt)
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= Int(gen_id)

		push!(range_df_ar, range_df)
	end
end

ess_cnt_df = vcat(range_df_ar...)
first(ess_cnt_df, 2)

Row,first,last,ess_cnt,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [13]:
first(is_pos_df, 2)

Row,start,end,IS_strand,length,IS_id,Line,Gen
,Float64,Float64,String,Float64,Float64,String,Float64
1,476928.0,480020.0,reverse,3093.0,0.0,L01-1,1.0
2,557167.0,560259.0,reverse,3093.0,1.0,L01-1,1.0


In [14]:
# is positions (used for pre-MA count)
lines = unique(ess_pos_df.line)
gen_ids = unique(ess_pos_df.gen_id)
buffer = 20
range_length = 10000

range_df_ar = []

is_pos_df.start = Int.(is_pos_df.start)
is_pos_df.end = Int.(is_pos_df.end)
is_pos_df.Gen = Int.(is_pos_df.Gen)

for line in lines
	for gen_id in gen_ids
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]
		
		n_range = trunc(Int, genome_size_ / range_length)
		df_ = is_pos_df[is_pos_df.Line .== line .&& is_pos_df.Gen .== gen_id,
		                 [:start, :end]]
		df_[!, :first] = min.(df_.start, df_.end) .- buffer
		df_[!, :last] = max.(df_.start, df_.end) .+ buffer

		# Generate range DataFrame
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)

		# Compute overlaps
		within_range!((range_df), df_)
		rename!(range_df, :overlap_count => :is_cnt)
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= Int(gen_id)

		push!(range_df_ar, range_df)
	end
end

pre_MA_cnt_df = vcat(range_df_ar...)
first(pre_MA_cnt_df, 2)

Row,first,last,is_cnt,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [15]:
first(ins_pos_prevgen, 2)

Row,pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
,Float64,Bool,String,Float64,String,Float64,Float64,Bool,String,String,String,Float64
1,0.0,false,f,476928.0,Ref,26.0,1.0,false,original,pristine,L01-1,2.0
2,1.0,false,f,557167.0,Ref,23.0,4.0,false,lost_fragment,unknown,L01-1,2.0


In [16]:
ins_pos_prevgen.position_status |> unique

3-element Vector{String}:
 "original"
 "lost_fragment"
 "new"

In [17]:
# insertion positions
lines = unique(ess_pos_df.line)
gen_ids = Int.(unique(ess_pos_df.gen_id))
buffer = 20
range_length = 10000

range_df_ar = []

# I am interested in the insertion events
ins_pos_prevgen = @chain ins_pos_prevgen begin
    @subset(:genome .== "Query", :position_status .== "new")
end
ins_pos_prevgen.pos = Int.(ins_pos_prevgen.pos)
ins_pos_prevgen.Gen = Int.(ins_pos_prevgen.Gen)

for line in lines
	for gen_id in [2, 3]
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]
		
		n_range = trunc(Int, genome_size_ / range_length)
		df_ = ins_pos_prevgen[ins_pos_prevgen.Line .== line .&& ins_pos_prevgen.Gen .== gen_id,
		                 [:pos, :insert_id]]
		print(size(df_)) # for check
		df_ = @chain df_ begin
			groupby(:insert_id) # unique positions in ref genome. to avoid double counting of insertion ends and insertions within tandem duplications.
			@combine(:pos = median(:pos))
		end
		df_[!, :first] = df_.pos .- buffer
		df_[!, :last] = df_.pos .+ buffer

		# Generate range DataFrame
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)

		# Compute overlaps
		within_range!((range_df), df_)
		rename!(range_df, :overlap_count => :ins_cnt)
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= gen_id - 1 # compare to ess_genes in previous genome

		push!(range_df_ar, range_df)
	end
end

ins_cnt_df = vcat(range_df_ar...)
first(ins_cnt_df, 2)

(15, 2)(34, 2)(15, 2)(16, 2)(26, 2)(14, 2)(17, 2)(20, 2)(19, 2)(24, 2)(22, 2)(8, 2)(23, 2)(9, 2)(28, 2)(25, 2)(10, 2)(14, 2)(21, 2)(11, 2)(26, 2)(21, 2)(25, 2)(19, 2)(22, 2)(21, 2)(16, 2)(24, 2)(24, 2)(14, 2)(22, 2)(24, 2)(22, 2)(10, 2)(33, 2)(29, 2)(25, 2)(13, 2)(28, 2)(26, 2)(19, 2)(20, 2)(35, 2)(10, 2)(24, 2)(15, 2)(28, 2)(9, 2)(11, 2)(3, 2)(7, 2)(7, 2)(10, 2)(5, 2)(5, 2)(17, 2)(8, 2)(4, 2)(3, 2)(20, 2)(4, 2)(3, 2)(5, 2)(5, 2)(6, 2)(9, 2)(6, 2)(7, 2)(12, 2)(17, 2)(5, 2)(2, 2)(6, 2)(12, 2)(25, 2)(3, 2)(11, 2)(20, 2)(11, 2)(24, 2)(10, 2)(9, 2)(17, 2)(11, 2)(3, 2)(5, 2)(22, 2)(12, 2)

Row,first,last,ins_cnt,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [18]:
# check: L01-1 that has duplications has windows couting only once for the insertion
@chain ins_cnt_df begin
    @subset(:line .== "L01-1", :gen_id .== 1)
    @subset(:ins_cnt .> 0, 1.5e6 .< :first .< 2e6)
    first(5)
end

Row,first,last,ins_cnt,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1670001,1680000,1,L01-1,1
2,1950001,1960000,1,L01-1,1


In [19]:
@chain ins_pos_prevgen begin
    @subset(:Line .== "L01-1", :Gen .== 2)
    @subset(1.5e6 .< :pos .< 2e6)
end

Row,pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
,Float64,Bool,String,Int64,String,Float64,Float64,Bool,String,String,String,Int64
1,7.0,false,r,1675574,Query,1.0,15.0,true,new,simple_insertion,L01-1,2
2,7.0,false,f,1675561,Query,1.0,15.0,false,new,simple_insertion,L01-1,2
3,8.0,true,f,1952972,Query,2.0,16.0,true,new,simple_insertion,L01-1,2
4,8.0,true,r,1952959,Query,2.0,16.0,false,new,simple_insertion,L01-1,2
5,9.0,true,r,1952959,Query,2.0,16.0,false,new,simple_insertion,L01-1,2


In [20]:
ins_pos_prevgen.sv |> unique

3-element Vector{String}:
 "unknown"
 "simple_insertion"
 "simple_transposition"

In [21]:
range_df_ar = []

ins_pos_prevgen_simple = @chain ins_pos_prevgen begin
    @subset(map(x -> occursin(r"simple", x), ins_pos_prevgen.sv))
end

for line in lines
	for gen_id in [2, 3]
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]

		# Generate range DataFrame
		n_range = trunc(Int, genome_size_ / range_length)
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)
		
		df_ = @chain ins_pos_prevgen_simple begin
                    @subset(:Line .== line, :Gen .== gen_id)   
                    @select(:pos, :insert_id)
                end

        if size(df_)[1] > 0
            df_ = @chain df_ begin
                groupby(:insert_id) # unique positions in ref genome. to avoid double counting of insertion ends and insertions within tandem duplications.
                @combine(:pos = median(:pos))
            end
            df_[!, :first] = df_.pos .- buffer
            df_[!, :last] = df_.pos .+ buffer

            # Compute overlaps
            within_range!((range_df), df_)
		    rename!(range_df, :overlap_count => :ins_cnt_simple)
        else
            range_df[!, :ins_cnt_simple] .= 0
        end
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= gen_id - 1 # compare to ess_genes in previous genome

		push!(range_df_ar, range_df)
	end
end

ins_cnt_simple_df = vcat(range_df_ar...)
first(ins_cnt_simple_df, 2)

Row,first,last,ins_cnt_simple,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [22]:
@chain ins_pos_prevgen begin
    @subset(:Line .== "L01-2", :Gen .== 3)
    @subset(2.5e6 .< :pos .< 3e6)
end

Row,pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
,Float64,Bool,String,Int64,String,Float64,Float64,Bool,String,String,String,Int64
1,13.0,true,f,2664982,Query,27.0,29.0,true,new,unknown,L01-2,3
2,13.0,true,r,2664941,Query,21.0,28.0,false,new,unknown,L01-2,3


In [23]:
# merge the dataframes and save
#ess_cnt_df, pre_MA_cnt_df, ins_cnt_df
cnt_df = @chain ess_cnt_df begin
    innerjoin(pre_MA_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_simple_df, on=[:line, :gen_id, :first, :last])
end
print(size(cnt_df))
cnt_df |> x -> first(x, 2)

(34985, 8)

Row,first,last,ess_cnt,line,gen_id,is_cnt,ins_cnt,ins_cnt_simple
,Int64,Int64,Int64,String,Int64,Int64,Int64,Int64
1,1,10000,0,L01-1,1,0,0,0
2,10001,20000,0,L01-1,1,0,0,0


In [24]:
cnt_df.gen_id |> unique

2-element Vector{Int64}:
 1
 2

In [25]:
# save full_data as arrow and non_zero as csv
@rget file_name_base dat_dir tmp_dir
cnt_df_non_zero = @chain cnt_df begin
    @subset((:ess_cnt .> 0) .| (:is_cnt .> 0) .| (:ins_cnt .> 0) .| (:ins_cnt_simple .> 0))
end
cnt_df_base_name_ = "$(file_name_base)_INS_IS_Ess-gene_cnt_in_$(range_length)"
CSV.write(joinpath(dat_dir, "$(cnt_df_base_name_).csv"), cnt_df_non_zero)
Arrow.write(joinpath(tmp_dir, "$(cnt_df_base_name_).arrow"), cnt_df; compress = :zstd)

"exp/multiple_runs14/tmp/R01_INS_ESS_corr/20250204__INS_IS_Ess-gene_cnt_in_10000.arrow"

In [26]:
R"""
# rm
rm(ess_pos_df, is_pos_df, ins_pos_prevgen, genome_size_df)
gc(); gc();
"""

RObject{RealSxp}
          used (Mb) gc trigger  (Mb) max used  (Mb)
Ncells 1248221 66.7    2438998 130.3  2438998 130.3
Vcells 2173687 16.6    8388608  64.0  4014754  30.7


## Statistical analysis

In [27]:
@rput cnt_df_base_name_
R"""
cnt_df_ar <- open_dataset(file.path(tmp_dir, paste0(cnt_df_base_name_, ".arrow")), format = "arrow")
cnt_df_ar %>% head(2) %>% collect
"""

RObject{VecSxp}
# A tibble: 2 × 8
  first  last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple
  <int> <int>   <int> <chr>  <int>  <int>   <int>          <int>
1     1 10000       0 L01-1      1      0       0              0
2 10001 20000       0 L01-1      1      0       0              0


In [28]:
R"""
cnt_df_ar %>% pull(ins_cnt, as_vector = T) %>% table()
"""

RObject{IntSxp}
.
    0     1     2     3 
34137   783    64     1 


In [29]:
R"""
cnt_df_ar %>% pull(ins_cnt_simple, as_vector = T) %>% table()
"""

RObject{IntSxp}
.
    0     1     2 
34616   368     1 


In [30]:
# look at the plotsr plot to see that the counts are plausible
# doule counts tne do be small deletions from both ends of ISs
R"""
cnt_df_ar %>% filter(ins_cnt == 2) %>% head %>% collect
"""

RObject{VecSxp}
# A tibble: 6 × 8
    first    last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple
    <int>   <int>   <int> <chr>  <int>  <int>   <int>          <int>
1 2660001 2670000       0 L01-1      1      0       2              0
2 1790001 1800000       1 L01-1      2      0       2              0
3 3520001 3530000       0 L01-2      1      0       2              0
4 2660001 2670000       0 L01-2      2      0       2              0
5 3470001 3480000       0 L01-3      1      0       2              0
6 1040001 1050000       0 L01-3      2      0       2              0


In [31]:
R"""
cnt_df_ar %>% filter(ins_cnt_simple >= 2) %>% head %>% collect
"""

RObject{VecSxp}
# A tibble: 1 × 8
   first   last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple
   <int>  <int>   <int> <chr>  <int>  <int>   <int>          <int>
1 850001 860000       0 L03-2      1      0       2              2


In [32]:
@chain ins_pos_prevgen begin # previously I was misassigning a large TSD as a double simple insertion
    @subset(:Line .== "L01-2", :Gen .== 3)
    @subset(2.5e6 .< :pos .< 3e6)
end

Row,pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
,Float64,Bool,String,Int64,String,Float64,Float64,Bool,String,String,String,Int64
1,13.0,true,f,2664982,Query,27.0,29.0,true,new,unknown,L01-2,3
2,13.0,true,r,2664941,Query,21.0,28.0,false,new,unknown,L01-2,3


In [33]:
@chain ins_pos_prevgen begin # there indeed are double simple insertions
    @subset(:Line .== "L03-2", :Gen .== 2)
    @subset(0.5e6 .< :pos .< 1e6)
end

Row,pos_id,IS_strand,flank,pos,genome,cluster_id,insert_id,flankMatchDir,position_status,sv,Line,Gen
,Float64,Bool,String,Int64,String,Float64,Float64,Bool,String,String,String,Int64
1,4.0,false,r,850970,Query,0.0,6.0,true,new,simple_insertion,L03-2,2
2,4.0,false,f,850956,Query,0.0,6.0,false,new,simple_insertion,L03-2,2
3,5.0,false,r,852452,Query,3.0,7.0,true,new,simple_insertion,L03-2,2
4,5.0,false,f,852439,Query,3.0,7.0,false,new,simple_insertion,L03-2,2


In [34]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt <- glm(ins_cnt ~ ess_cnt + gen_id + is_cnt,
    data = cnt_df_ar %>% collect, family = poisson(link = "log"))

summary(glm.ins_cnt) %>% print
Anova(glm.ins_cnt, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt ~ ess_cnt + gen_id + is_cnt, family = poisson(link = "log"), 
    data = cnt_df_ar %>% collect)

Coefficients:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept) -3.41203    0.10450 -32.652  < 2e-16 ***
ess_cnt     -0.19331    0.03279  -5.895 3.75e-09 ***
gen_id      -0.17665    0.06657  -2.654  0.00796 ** 
is_cnt       1.38123    0.08409  16.426  < 2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 6846.8  on 34984  degrees of freedom
Residual deviance: 6582.1  on 34981  degrees of freedom
AIC: 8326.4

Number of Fisher Scoring iterations: 6

Analysis of Deviance Table (Type III tests)

Response: ins_cnt
        LR Chisq Df Pr(>Chisq)    
ess_cnt   47.610  1    5.2e-12 ***
gen_id     7.055  1   0.007903 ** 
is_cnt   197.962  1  < 2.2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
        LR Chisq Df Pr(>Chisq)    
ess_cnt   47.610  1    5.2e-12 ***
gen_id     7.055  1   0.007903 ** 
is_cnt   197.962  1  < 2.2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [35]:
R"""
glm.ins_cnt_simple <- glm(ins_cnt_simple ~ ess_cnt + gen_id + is_cnt,
    data = cnt_df_ar %>% collect, family = poisson(link = "log"))

summary(glm.ins_cnt_simple) %>% print
Anova(glm.ins_cnt_simple, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt_simple ~ ess_cnt + gen_id + is_cnt, family = poisson(link = "log"), 
    data = cnt_df_ar %>% collect)

Coefficients:
            Estimate Std. Error z value Pr(>|z|)    
(Intercept) -4.03760    0.16106 -25.069  < 2e-16 ***
ess_cnt     -0.16985    0.04676  -3.632 0.000281 ***
gen_id      -0.24438    0.10489  -2.330 0.019811 *  
is_cnt      -1.62099    0.50264  -3.225 0.001260 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 3369.2  on 34984  degrees of freedom
Residual deviance: 3328.0  on 34981  degrees of freedom
AIC: 4074.6

Number of Fisher Scoring iterations: 8

Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
        LR Chisq Df Pr(>Chisq)    
ess_cnt  17.7124  1  2.569e-05 ***
gen_id    5.4727  1    0.01932 *  
is_cnt   18.8275  1  1.431e-05 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
        LR Chisq Df Pr(>Chisq)    
ess_cnt  17.7124  1  2.569e-05 ***
gen_id    5.4727  1    0.01932 *  
is_cnt   18.8275  1  1.431e-05 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


# Per non-essential locus
I should have compared with the rate per non-essential locus.
The effect of essential genes seem to be ~0.25 and very close to the size of the gene within the window ~0.2.

In [36]:
function count_overlap_bases!(range_df::DataFrame, pos_df::DataFrame, range_length::Int)
    # Step 1: Initiaze per base vector
    pos_vec = zeros(Int, range_length)

    # Step 2: Mark positions covered by pos_df as 1
    for i in 1:nrow(pos_df)
        pf, pe = pos_df.first[i], pos_df.last[i]  # Start and end of pos_df range
        pf, pe = Int(pf), Int(pe)
        pos_vec[pf:pe] .= 1  # Mark covered bases
    end

    # Step 3: Count overlapping bases per range in range_df
    range_df[!, :overlap_bases] .= 0  # Initialize the new column

    for i in 1:nrow(range_df)
        rf, re = range_df.first[i], range_df.last[i]  # Start and end of range_df range
        range_df.overlap_bases[i] = sum(pos_vec[rf:re])  # Sum marked bases in the range
    end

    return range_df
end

count_overlap_bases! (generic function with 1 method)

In [37]:
range_df_ar = []

for line in lines
	for gen_id in gen_ids
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]
		
		n_range = trunc(Int, genome_size_ / range_length)
		df_ = ess_pos_df[ess_pos_df.line .== line .&& ess_pos_df.gen_id .== gen_id,
		                 [:sstart, :send]]
		# no buffer
		df_[!, :first] = min.(df_.sstart, df_.send)
		df_[!, :last] = max.(df_.sstart, df_.send) 

		# Generate range DataFrame
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)

		# Compute overlaps
        count_overlap_bases!(range_df, df_, Int(genome_size_))
		rename!(range_df, :overlap_bases => :ess_bp)
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= Int(gen_id)

		push!(range_df_ar, range_df)
	end
end

ess_bp_df = vcat(range_df_ar...)
first(ess_bp_df, 2)

Row,first,last,ess_bp,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [38]:
first(ess_bp_df[ess_bp_df.ess_bp .> 0, :], 3)

Row,first,last,ess_bp,line,gen_id
,Int64,Int64,Int64,String,Int64
1,20001,30000,75,L01-1,1
2,40001,50000,1259,L01-1,1
3,50001,60000,228,L01-1,1


In [39]:
(@chain ess_pos_df begin # check if ranges are sensible
    @subset(:line .== "L01-1", :gen_id .== 1)
    sort(:sstart)
end)|> x -> first(x, 5)

Row,gene,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,line,gen_id,duplicated
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64,Bool
1,trpT,100.0,75.0,0.0,0.0,1.0,75.0,21215.0,21289.0,2.09e-34,139.0,75.0,L01-1,1.0,false
2,rho,100.0,1259.0,0.0,0.0,1.0,1259.0,40675.0,41933.0,0.0,2326.0,1259.0,L01-1,1.0,false
3,argX,100.0,76.0,0.0,0.0,1.0,76.0,56633.0,56708.0,5.93e-35,141.0,76.0,L01-1,1.0,false
4,hisR,100.0,76.0,0.0,0.0,1.0,76.0,56767.0,56842.0,5.93e-35,141.0,76.0,L01-1,1.0,false
5,proM,100.0,76.0,0.0,0.0,1.0,76.0,56992.0,57067.0,5.93e-35,141.0,76.0,L01-1,1.0,false


In [40]:
# some ranges that are completely covered by essential genes
ess_bp_df[ess_bp_df.ess_bp .== range_length, :] |> x -> first(x, 3) 

Row,first,last,ess_bp,line,gen_id
,Int64,Int64,Int64,String,Int64


In [41]:
(@chain ess_pos_df begin # check if the complete coverage is plausible -> mukB is large
    @subset(:line .== "L02-4", :gen_id .== 2)
    sort(:sstart)
    @subset(1.468e6 .< :sstart .< 1.5e6)
end)|> x -> first(x, 5)

Row,gene,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,line,gen_id,duplicated
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64,Bool
1,mukF,100.0,1322.0,0.0,0.0,1.0,1322.0,1.46862e6,1.46994e6,0.0,2442.0,1322.0,L02-4,2.0,false
2,mukE,100.0,677.0,0.0,0.0,1.0,677.0,1.46995e6,1.47062e6,0.0,1251.0,677.0,L02-4,2.0,false
3,mukB,100.0,4460.0,0.0,0.0,1.0,4460.0,1.47063e6,1.47508e6,0.0,8237.0,4460.0,L02-4,2.0,false
4,asnS,100.0,1400.0,0.0,0.0,1.0,1400.0,1.48188e6,1.48328e6,0.0,2586.0,1400.0,L02-4,2.0,false


In [42]:
(@chain ess_pos_df begin # check if the complete coverage is plausible -> ftsI, murE overlap
    @subset(:line .== "L03-2", :gen_id .== 2)
    sort(:sstart)
    @subset(0.715e6 .< :sstart .< 1.5e6)
end)|> x -> first(x, 5)

Row,gene,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,line,gen_id,duplicated
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64,Bool
1,ftsL,100.0,365.0,0.0,0.0,1.0,365.0,719459.0,719823.0,0.0,675.0,365.0,L03-2,2.0,false
2,ftsI,100.0,1766.0,0.0,0.0,1.0,1766.0,719840.0,721605.0,0.0,3262.0,1766.0,L03-2,2.0,false
3,murE,100.0,1487.0,0.0,0.0,1.0,1487.0,721593.0,723079.0,0.0,2747.0,1487.0,L03-2,2.0,false
4,murF,100.0,1358.0,0.0,0.0,1.0,1358.0,723077.0,724434.0,0.0,2508.0,1358.0,L03-2,2.0,false
5,mraY,100.0,1082.0,0.0,0.0,1.0,1082.0,724429.0,725510.0,0.0,1999.0,1082.0,L03-2,2.0,false


In [43]:
# merge the dataframes and save
#ess_cnt_df, pre_MA_cnt_df, ins_cnt_df
cnt_df = @chain ess_cnt_df begin
    innerjoin(pre_MA_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_simple_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ess_bp_df, on=[:line, :gen_id, :first, :last])
end
print(size(cnt_df))
cnt_df |> x -> first(x, 2)

(34985, 9)

Row,first,last,ess_cnt,line,gen_id,is_cnt,ins_cnt,ins_cnt_simple,ess_bp
,Int64,Int64,Int64,String,Int64,Int64,Int64,Int64,Int64
1,1,10000,0,L01-1,1,0,0,0,0
2,10001,20000,0,L01-1,1,0,0,0,0


In [44]:
# save full_data as arrow and non_zero as csv
@rget file_name_base dat_dir tmp_dir
cnt_df_non_zero = @chain cnt_df begin
    @subset((:ess_cnt .> 0) .| (:is_cnt .> 0) .| (:ins_cnt .> 0) .| (:ins_cnt_simple .> 0) .| (:ess_bp .> 0))
end
cnt_df_base_name_ = "$(file_name_base)_INS_IS_Ess-gene_cnt_in_$(range_length)"
CSV.write(joinpath(dat_dir, "$(cnt_df_base_name_).csv"), cnt_df_non_zero)
Arrow.write(joinpath(tmp_dir, "$(cnt_df_base_name_).arrow"), cnt_df; compress = :zstd)

"exp/multiple_runs14/tmp/R01_INS_ESS_corr/20250204__INS_IS_Ess-gene_cnt_in_10000.arrow"

## Statistical analysis2

In [45]:
@rput cnt_df_base_name_
R"""
cnt_df_ar <- open_dataset(file.path(tmp_dir, paste0(cnt_df_base_name_, ".arrow")), format = "arrow")
cnt_df_ar %>% head(2) %>% collect
"""

RObject{VecSxp}
# A tibble: 2 × 9
  first  last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple ess_bp
  <int> <int>   <int> <chr>  <int>  <int>   <int>          <int>  <int>
1     1 10000       0 L01-1      1      0       0              0      0
2 10001 20000       0 L01-1      1      0       0              0      0


In [46]:
R"""
cnt_df_ar %>% filter(ess_bp > 0) %>% head %>% collect %>%
    mutate(non_ess_bp = last - first + 1 - ess_bp) %>% select(-ins_cnt_simple)
"""

RObject{VecSxp}
# A tibble: 6 × 9
   first   last ess_cnt line  gen_id is_cnt ins_cnt ess_bp non_ess_bp
   <int>  <int>   <int> <chr>  <int>  <int>   <int>  <int>      <dbl>
1  20001  30000       1 L01-1      1      0       0     75       9925
2  40001  50000       1 L01-1      1      0       0   1259       8741
3  50001  60000       3 L01-1      1      0       0    228       9772
4  60001  70000       2 L01-1      1      0       0   1678       8322
5 100001 110000       1 L01-1      1      0       0    545       9455
6 120001 130000       2 L01-1      1      0       0   3382       6618


In [47]:
R"""
cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_bp) %>% pull(non_ess_bp, as_vector = TRUE) %>% summary 
"""

RObject{RealSxp}
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
     57    9173   10000    9233   10000   10000 


In [48]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt <- glm(ins_cnt ~ is_cnt + ess_cnt, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt) %>% print
Anova(glm.ins_cnt, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt ~ is_cnt + ess_cnt, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_bp) %>% mutate(passages = ifelse(gen_id == 1, 8, 
        12)) %>% filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.18638    0.03998 -379.860   <2e-16 ***
is_cnt        1.31104    0.08350   15.700   <2e-16 ***
ess_cnt      -0.07109    0.03309   -2.148   0.0317 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 6842.9  on 34984  degrees of freedom
Residual deviance: 6650.3  on 34982  degrees of freedom
AIC: 8392.6

Number of Fisher Scoring iterations: 7

Analysis of Deviance Table (Type III tests)

Response: ins_cnt
        LR Chisq Df Pr(>Chisq)    
is_cnt   182.415  1    < 2e-16 ***
ess_cnt    5.099  1    0.02394 *  

RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
        LR Chisq Df Pr(>Chisq)    
is_cnt   182.415  1    < 2e-16 ***
ess_cnt    5.099  1    0.02394 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [49]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt_simple <- glm(ins_cnt_simple ~ is_cnt + ess_cnt, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt_simple) %>% print
Anova(glm.ins_cnt_simple, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt_simple ~ is_cnt + ess_cnt, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_bp) %>% mutate(passages = ifelse(gen_id == 1, 8, 
        12)) %>% filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.91098    0.05773 -275.592  < 2e-16 ***
is_cnt       -1.69485    0.50244   -3.373 0.000743 ***
ess_cnt      -0.04572    0.04725   -0.968 0.333245    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 3382.5  on 34984  degrees of freedom
Residual deviance: 3360.6  on 34982  degrees of freedom
AIC: 4105.3

Number of Fisher Scoring iterations: 8

Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
        LR Chisq Df Pr(>Chisq)    
is_cnt   21.2663  1  3.997e-06 ***
ess_cnt   1.0014  1 

RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
        LR Chisq Df Pr(>Chisq)    
is_cnt   21.2663  1  3.997e-06 ***
ess_cnt   1.0014  1      0.317    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


## Use binary: the presence of essential genes
Having counts as predictor might not be good, as that would have higher values for shorter genes like tRNA.
Analyze with only if there is an essential gene or not.

Not only that is more sensible, but also the AIC is slightly better with the binary predictor.

In [50]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt <- glm(ins_cnt ~ is_cnt + ess_exist, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt) %>% print
Anova(glm.ins_cnt, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt ~ is_cnt + ess_exist, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_bp) %>% mutate(passages = ifelse(gen_id == 1, 8, 
        12)) %>% mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>% 
        filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.17189    0.04193 -361.807   <2e-16 ***
is_cnt        1.30665    0.08363   15.623   <2e-16 ***
ess_exist    -0.18444    0.07745   -2.381   0.0172 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 6842.9  on 34984  degrees of freedom
Residual deviance: 6649.5  on 34982  degrees of freedom
AIC: 8391.8

Number of Fisher Scoring iterations: 6

Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt

RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     181.072  1    < 2e-16 ***
ess_exist    5.843  1    0.01564 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [51]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt_simple <- glm(ins_cnt_simple ~ is_cnt + ess_exist, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt_simple) %>% print
Anova(glm.ins_cnt_simple, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt_simple ~ is_cnt + ess_exist, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_bp) %>% mutate(passages = ifelse(gen_id == 1, 8, 
        12)) %>% mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>% 
        filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.92217    0.06211 -256.357  < 2e-16 ***
is_cnt       -1.68886    0.50251   -3.361 0.000777 ***
ess_exist    -0.04663    0.11440   -0.408 0.683581    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 3382.5  on 34984  degrees of freedom
Residual deviance: 3361.5  on 34982  degrees of freedom
AIC: 4106.1

Number of Fisher Scoring iterations: 8

Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chi

RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     21.0449  1  4.487e-06 ***
ess_exist   0.1672  1     0.6826    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [52]:
R""" # convert coefficients to fold change
glm.ins_cnt$coefficients %>% exp %>% print
glm.ins_cnt_simple$coefficients %>% exp %>% print
""";

 (Intercept)       is_cnt    ess_exist 
2.575919e-07 3.693763e+00 8.315698e-01 
 (Intercept)       is_cnt    ess_exist 
1.216442e-07 1.847294e-01 9.544441e-01 


In [53]:
R"""
cnt_df_ar %>% summarise(sum(ins_cnt), sum(ins_cnt_simple)) %>% collect
"""

RObject{VecSxp}
# A tibble: 1 × 2
  `sum(ins_cnt)` `sum(ins_cnt_simple)`
           <int>                 <int>
1            914                   370


In [54]:
# check that the model is returning a sensible value 
R"""
# bp * passages * intercept * ranges/genome * genomes
 10 * exp(glm.ins_cnt_simple$coefficient)["(Intercept)"] * (4e6) * (44 * 2)
"""

RObject{RealSxp}
(Intercept) 
   428.1876 


# Per non-essential and non-IS locus
The above analysis indicate a very strong negative correlation between simple insertions and pre-existing IS.
This may partially be due to insertions within ISs not being counted.
Here, I calculate the non-essential and non-IS bp count to factor out the effect of ISs.

In [55]:
range_df_ar = []

for line in lines
	for gen_id in gen_ids
		genome_size_ = @chain genome_size_df begin
		    @subset(:Prefix .== line, :gen_id .== gen_id)
		end
		genome_size_ = genome_size_.genome_size[1]
		
		n_range = trunc(Int, genome_size_ / range_length)
		df_ = ess_pos_df[ess_pos_df.line .== line .&& ess_pos_df.gen_id .== gen_id,
		                 [:sstart, :send]]
		# no buffer
		df_[!, :first] = min.(df_.sstart, df_.send)
		df_[!, :last] = max.(df_.sstart, df_.send) 

		## add the IS positions
		df_2 = is_pos_df[is_pos_df.Line .== line .&& is_pos_df.Gen .== gen_id,
		                 [:start, :end]]
		df_2[!, :first] = min.(df_2.start, df_2.end)
		df_2[!, :last] = max.(df_2.start, df_2.end)

		# cat the dfs
		df_ = df_[:, [:first, :last]]
		df_2 = df_2[:, [:first, :last]]
		df_ = vcat(df_, df_2)

		# Generate range DataFrame
		range_df = DataFrame(first=collect(1:range_length:(1 + range_length * (n_range - 1))),
		                      last=collect(range_length:range_length:genome_size_))
		# int
		range_df.first = Int.(range_df.first)
		range_df.last = Int.(range_df.last)

		# Compute overlaps
        count_overlap_bases!(range_df, df_, Int(genome_size_))
		rename!(range_df, :overlap_bases => :ess_IS_bp)
		range_df[!, :line] .= line
		range_df[!, :gen_id] .= Int(gen_id)

		push!(range_df_ar, range_df)
	end
end

ess_is_bp_df = vcat(range_df_ar...)
first(ess_is_bp_df, 2)

Row,first,last,ess_IS_bp,line,gen_id
,Int64,Int64,Int64,String,Int64
1,1,10000,0,L01-1,1
2,10001,20000,0,L01-1,1


In [56]:
first(is_pos_df, 2)

Row,start,end,IS_strand,length,IS_id,Line,Gen
,Int64,Int64,String,Float64,Float64,String,Int64
1,476928,480020,reverse,3093.0,0.0,L01-1,1
2,557167,560259,reverse,3093.0,1.0,L01-1,1


In [57]:
range_start = is_pos_df.start[1]
range_start = round(range_start / range_length) * range_length
@chain ess_is_bp_df begin
	@subset(:line .== "L01-1", :gen_id .== 1)
	@subset(range_start .< :first .< range_start + range_length*2)
end

Row,first,last,ess_IS_bp,line,gen_id
,Int64,Int64,Int64,String,Int64
1,480001,490000,20,L01-1,1
2,490001,500000,0,L01-1,1


In [58]:
(@chain ess_pos_df begin # check if ranges are sensible
    @subset(:line .== "L01-1", :gen_id .== 1)
    @subset(range_start .< :sstart .< range_start + range_length*2)
end)

Row,gene,pident,length,mismatch,gapopen,qstart,qend,sstart,send,evalue,bitscore,qlen,line,gen_id,duplicated
,String,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,Float64,String,Float64,Bool


In [59]:
# merge the dataframes and save
#ess_cnt_df, pre_MA_cnt_df, ins_cnt_df
cnt_df = @chain ess_cnt_df begin
    innerjoin(pre_MA_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ins_cnt_simple_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ess_bp_df, on=[:line, :gen_id, :first, :last])
    innerjoin(ess_is_bp_df, on=[:line, :gen_id, :first, :last])
end
print(size(cnt_df))
cnt_df |> x -> first(x, 2)

(34985, 10)

Row,first,last,ess_cnt,line,gen_id,is_cnt,ins_cnt,ins_cnt_simple,ess_bp,ess_IS_bp
,Int64,Int64,Int64,String,Int64,Int64,Int64,Int64,Int64,Int64
1,1,10000,0,L01-1,1,0,0,0,0,0
2,10001,20000,0,L01-1,1,0,0,0,0,0


In [60]:
# check that ess_bp <= ess_is_bp. Should be empty.
@chain cnt_df begin
	@subset(:ess_bp .> :ess_IS_bp)
	@select(:line, :gen_id, :first, :last, :ess_bp, :ess_IS_bp)
end

Row,line,gen_id,first,last,ess_bp,ess_IS_bp
,String,Int64,Int64,Int64,Int64,Int64


In [61]:
# check that ess_bp <= ess_is_bp. Should be empty.
(@chain cnt_df begin
	@subset(:ess_bp .!= :ess_IS_bp)
	@select(:line, :gen_id, :first, :last, :ess_bp, :ess_IS_bp)
end) |> x -> first(x, 4)

Row,line,gen_id,first,last,ess_bp,ess_IS_bp
,String,Int64,Int64,Int64,Int64,Int64
1,L01-1,1,470001,480000,186,3259
2,L01-1,1,480001,490000,0,20
3,L01-1,1,550001,560000,0,2834
4,L01-1,1,560001,570000,5037,5296


In [62]:
# save full_data as arrow and non_zero as csv
@rget file_name_base dat_dir tmp_dir
cnt_df_non_zero = @chain cnt_df begin
    @subset((:ess_cnt .> 0) .| (:is_cnt .> 0) .| (:ins_cnt .> 0) .| (:ins_cnt_simple .> 0) .| (:ess_bp .> 0))
end
cnt_df_base_name_ = "$(file_name_base)_INS_IS_Ess-gene_cnt_in_$(range_length)"
CSV.write(joinpath(dat_dir, "$(cnt_df_base_name_).csv"), cnt_df_non_zero)
Arrow.write(joinpath(tmp_dir, "$(cnt_df_base_name_).arrow"), cnt_df; compress = :zstd)

"exp/multiple_runs14/tmp/R01_INS_ESS_corr/20250204__INS_IS_Ess-gene_cnt_in_10000.arrow"

## Use binary: the presence of essential genes
Do the same analysis with the non-essential and non-IS bp count.
AIC became slightly better factoring out the effect of ISs.

In [63]:
@rput cnt_df_base_name_
R"""
cnt_df_ar <- open_dataset(file.path(tmp_dir, paste0(cnt_df_base_name_, ".arrow")), format = "arrow")
cnt_df_ar %>% head(2) %>% collect
"""

RObject{VecSxp}
# A tibble: 2 × 10
  first  last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple ess_bp
  <int> <int>   <int> <chr>  <int>  <int>   <int>          <int>  <int>
1     1 10000       0 L01-1      1      0       0              0      0
2 10001 20000       0 L01-1      1      0       0              0      0
# ℹ 1 more variable: ess_IS_bp <int>


In [64]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt <- glm(ins_cnt ~ is_cnt + ess_exist, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_IS_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt) %>% print
check_overdispersion(glm.ins_cnt) %>% print
check_zeroinflation(glm.ins_cnt) %>% print

Anova(glm.ins_cnt, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt ~ is_cnt + ess_exist, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_IS_bp) %>% mutate(passages = ifelse(gen_id == 1, 
        8, 12)) %>% mutate(ess_exist = ifelse(ess_cnt > 0, 1, 
        0)) %>% filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error z value Pr(>|z|)    
(Intercept) -15.17318    0.04198 -361.46   <2e-16 ***
is_cnt        1.59528    0.08362   19.08   <2e-16 ***
ess_exist    -0.17902    0.07749   -2.31   0.0209 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 6928.3  on 34983  degrees of freedom
Residual deviance: 6662.2  on 34981  degrees of freedom
AIC: 8404.4

Number of Fisher Scoring iterations: 6

# Overdispersion test

       dispersion ratio =     1.166
  Pearson's Chi-Squared = 40780.977
            

┌ Warning: RCall.jl: Overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


# Check for zero-inflation

   Observed zeros: 34136
  Predicted zeros: 34088
            Ratio: 1.00



┌ Warning: RCall.jl: Model seems ok, ratio of observed and predicted zeros is within the
│   tolerance range.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     253.228  1     <2e-16 ***
ess_exist    5.493  1     0.0191 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     253.228  1     <2e-16 ***
ess_exist    5.493  1     0.0191 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [65]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt_simple <- glm(ins_cnt_simple ~ is_cnt + ess_exist, 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_IS_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect, # otherwise the offset is singular
    family = poisson(link = "log"),
    offset = log(non_ess_bp*passages))

summary(glm.ins_cnt_simple) %>% print
check_overdispersion(glm.ins_cnt_simple) %>% print
check_zeroinflation(glm.ins_cnt_simple) %>% print

Anova(glm.ins_cnt_simple, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt_simple ~ is_cnt + ess_exist, family = poisson(link = "log"), 
    data = cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - 
        ess_IS_bp) %>% mutate(passages = ifelse(gen_id == 1, 
        8, 12)) %>% mutate(ess_exist = ifelse(ess_cnt > 0, 1, 
        0)) %>% filter(non_ess_bp > 0) %>% collect, offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.92225    0.06211 -256.341  < 2e-16 ***
is_cnt       -1.39867    0.50251   -2.783  0.00538 ** 
ess_exist    -0.04633    0.11440   -0.405  0.68546    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for poisson family taken to be 1)

    Null deviance: 3375.4  on 34983  degrees of freedom
Residual deviance: 3362.5  on 34981  degrees of freedom
AIC: 4107.1

Number of Fisher Scoring iterations: 8

# Overdispersion test

       dispersion ratio =     1.070
  Pearson's Chi-Squared = 37432.033
 

┌ Warning: RCall.jl: Overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172
┌ Warning: RCall.jl: Model seems ok, ratio of observed and predicted zeros is within the
│   tolerance range.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.8469  1   0.000338 ***
ess_exist   0.1651  1   0.684518    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.8469  1   0.000338 ***
ess_exist   0.1651  1   0.684518    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [66]:
R""" # convert coefficients to fold change
glm.ins_cnt$coefficients %>% exp %>% print
glm.ins_cnt_simple$coefficients %>% exp %>% print
""";

 (Intercept)       is_cnt    ess_exist 
2.572598e-07 4.929687e+00 8.360885e-01 
 (Intercept)       is_cnt    ess_exist 
1.216338e-07 2.469243e-01 9.547224e-01 


In [67]:
# check that the model is returning a sensible value 
R"""
# bp * passages * intercept * ranges/genome * genomes
5000 * 10 * exp(glm.ins_cnt_simple$coefficient)["(Intercept)"] * (4e6/5000) * (44 * 2)
"""

RObject{RealSxp}
(Intercept) 
   428.1511 


## poisson -> negative binomial / quasi poisson
care about the overdispersion. I end up using quasi poisson because the negative binomial did not converge for the simple insertions (might be due to too less non-zero data).

In [68]:
R"""
cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_IS_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect %>% # otherwise the offset is singular
        mutate(offset = log(non_ess_bp*passages)) %>%
		pull(offset) %>% summary
"""

RObject{RealSxp}
   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
  6.528  11.290  11.290  11.369  11.695  11.695 


In [69]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
tmp_dat_ <- cnt_df_ar %>% mutate(non_ess_bp = last - first + 1 - ess_IS_bp) %>%
        mutate(passages = ifelse(gen_id == 1, 8, 12)) %>%
        mutate(ess_exist = ifelse(ess_cnt > 0, 1, 0)) %>%
        filter(non_ess_bp > 0) %>% collect %>% # otherwise the offset is singular
        mutate(offset = log(non_ess_bp*passages))

glm.ins_cnt.nb <- glm.nb(ins_cnt ~ is_cnt + ess_exist + offset(log(non_ess_bp*passages)),
    data = tmp_dat_)

summary(glm.ins_cnt.nb) %>% print
check_overdispersion(glm.ins_cnt.nb) %>% print

Anova(glm.ins_cnt.nb, type = "III") %>% print
"""


Call:
glm.nb(formula = ins_cnt ~ is_cnt + ess_exist + offset(log(non_ess_bp * 
    passages)), data = tmp_dat_, init.theta = 0.2520606286, link = log)

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.16924    0.04418 -343.382   <2e-16 ***
is_cnt        1.65328    0.09639   17.152   <2e-16 ***
ess_exist    -0.18875    0.08141   -2.319   0.0204 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for Negative Binomial(0.2521) family taken to be 1)

    Null deviance: 5049.6  on 34983  degrees of freedom
Residual deviance: 4809.0  on 34981  degrees of freedom
AIC: 8303.2

Number of Fisher Scoring iterations: 1


              Theta:  0.2521 
          Std. Err.:  0.0408 

 2 x log-likelihood:  -8295.2360 
# Overdispersion test

 dispersion ratio = 0.949
          p-value = 0.384



┌ Warning: RCall.jl: No overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     228.386  1    < 2e-16 ***
ess_exist    5.504  1    0.01897 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     228.386  1    < 2e-16 ***
ess_exist    5.504  1    0.01897 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [70]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt.qp <- glm(ins_cnt ~ is_cnt + ess_exist,
    data = tmp_dat_, family = quasipoisson(link = "log"),
	offset = log(non_ess_bp*passages))

summary(glm.ins_cnt.qp) %>% print
check_overdispersion(glm.ins_cnt.qp) %>% print

Anova(glm.ins_cnt.qp, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt ~ is_cnt + ess_exist, family = quasipoisson(link = "log"), 
    data = tmp_dat_, offset = log(non_ess_bp * passages))

Coefficients:
             Estimate Std. Error t value Pr(>|t|)    
(Intercept) -15.17318    0.04532 -334.77   <2e-16 ***
is_cnt        1.59528    0.09029   17.67   <2e-16 ***
ess_exist    -0.17902    0.08367   -2.14   0.0324 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for quasipoisson family taken to be 1.165834)

    Null deviance: 6928.3  on 34983  degrees of freedom
Residual deviance: 6662.2  on 34981  degrees of freedom
AIC: NA

Number of Fisher Scoring iterations: 6

# Overdispersion test

       dispersion ratio =     1.166
  Pearson's Chi-Squared = 40780.977
                p-value =   < 0.001



┌ Warning: RCall.jl: Overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     217.208  1    < 2e-16 ***
ess_exist    4.711  1    0.02996 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt
          LR Chisq Df Pr(>Chisq)    
is_cnt     217.208  1    < 2e-16 ***
ess_exist    4.711  1    0.02996 *  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [71]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt_simple.nb <- glm.nb(ins_cnt_simple ~ is_cnt + ess_exist + offset(log(non_ess_bp*passages)),
    data = tmp_dat_, init.theta = 30)

summary(glm.ins_cnt_simple.nb) %>% print
check_overdispersion(glm.ins_cnt_simple.nb) %>% print

Anova(glm.ins_cnt_simple.nb, type = "III") %>% print
"""


Call:
glm.nb(formula = ins_cnt_simple ~ is_cnt + ess_exist + offset(log(non_ess_bp * 
    passages)), data = tmp_dat_, init.theta = 103.3058146, link = log)

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.92224    0.06212 -256.328  < 2e-16 ***
is_cnt       -1.39867    0.50251   -2.783  0.00538 ** 
ess_exist    -0.04634    0.11441   -0.405  0.68544    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for Negative Binomial(103.3058) family taken to be 1)

    Null deviance: 3371.8  on 34983  degrees of freedom
Residual deviance: 3358.9  on 34981  degrees of freedom
AIC: 4109.1

Number of Fisher Scoring iterations: 1


              Theta:  103 
          Std. Err.:  414 
Warning while fitting theta: iteration limit reached 

 2 x log-likelihood:  -4101.091 


┌ Warning: RCall.jl: Warning in theta.ml(Y, mu, sum(w), w, limit = control$maxit, trace = control$trace >  :
│   iteration limit reached
│ Warning in theta.ml(Y, mu, sum(w), w, limit = control$maxit, trace = control$trace >  :
│   iteration limit reached
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


# Overdispersion test

 dispersion ratio = 0.993
          p-value = 0.784

Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.8460  1  0.0003382 ***
ess_exist   0.1651  1  0.6844995    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


┌ Warning: RCall.jl: No overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.8460  1  0.0003382 ***
ess_exist   0.1651  1  0.6844995    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [72]:
# ins_cnt ~ ess_cnt gen_id is_cnt
R"""
glm.ins_cnt_simple.qp <- glm(ins_cnt_simple ~ is_cnt + ess_exist,
    data = tmp_dat_, family = quasipoisson(link = "log"),
	offset = log(non_ess_bp*passages))

summary(glm.ins_cnt_simple.qp) %>% print
check_overdispersion(glm.ins_cnt_simple.qp) %>% print

Anova(glm.ins_cnt_simple.qp, type = "III") %>% print
"""


Call:
glm(formula = ins_cnt_simple ~ is_cnt + ess_exist, family = quasipoisson(link = "log"), 
    data = tmp_dat_, offset = log(non_ess_bp * passages))

Coefficients:
             Estimate Std. Error  t value Pr(>|t|)    
(Intercept) -15.92225    0.06425 -247.806  < 2e-16 ***
is_cnt       -1.39867    0.51981   -2.691  0.00713 ** 
ess_exist    -0.04633    0.11834   -0.392  0.69541    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for quasipoisson family taken to be 1.070068)

    Null deviance: 3375.4  on 34983  degrees of freedom
Residual deviance: 3362.5  on 34981  degrees of freedom
AIC: NA

Number of Fisher Scoring iterations: 8

# Overdispersion test

       dispersion ratio =     1.070
  Pearson's Chi-Squared = 37432.033
                p-value =   < 0.001



┌ Warning: RCall.jl: Overdispersion detected.
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.0057  1  0.0005304 ***
ess_exist   0.1543  1  0.6944829    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


RObject{VecSxp}
Analysis of Deviance Table (Type III tests)

Response: ins_cnt_simple
          LR Chisq Df Pr(>Chisq)    
is_cnt     12.0057  1  0.0005304 ***
ess_exist   0.1543  1  0.6944829    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [73]:
R""" # convert coefficients to fold change
glm.ins_cnt.nb$coefficients %>% exp %>% print
glm.ins_cnt_simple.nb$coefficients %>% exp %>% print
""";

 (Intercept)       is_cnt    ess_exist 
2.582765e-07 5.224099e+00 8.279906e-01 
 (Intercept)       is_cnt    ess_exist 
1.216348e-07 2.469248e-01 9.547174e-01 


In [74]:
R"""
confint(glm.ins_cnt.nb) %>% exp %>% print
confint(glm.ins_cnt_simple.nb) %>% exp %>% print
""";

                   2.5 %       97.5 %
(Intercept) 2.365899e-07 2.814024e-07
is_cnt      4.298149e+00 6.327123e+00
ess_exist   7.045952e-01 9.697009e-01


┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


                   2.5 %       97.5 %
(Intercept) 1.074215e-07 1.370512e-07
is_cnt      7.637766e-02 5.773868e-01
ess_exist   7.600951e-01 1.190835e+00


┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


In [75]:
R""" # convert coefficients to fold change
glm.ins_cnt.qp$coefficients %>% exp %>% print
glm.ins_cnt_simple.qp$coefficients %>% exp %>% print
""";

 (Intercept)       is_cnt    ess_exist 
2.572598e-07 4.929687e+00 8.360885e-01 
 (Intercept)       is_cnt    ess_exist 
1.216338e-07 2.469243e-01 9.547224e-01 


In [76]:
R"""
confint(glm.ins_cnt.qp) %>% exp %>% print
confint(glm.ins_cnt_simple.qp) %>% exp %>% print
""";

                   2.5 %       97.5 %
(Intercept) 2.350955e-07 2.808139e-07
is_cnt      4.114951e+00 5.863868e+00
ess_exist   7.079696e-01 9.829913e-01


┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


                   2.5 %       97.5 %
(Intercept) 1.069529e-07 1.376019e-07
is_cnt      7.277449e-02 5.920655e-01
ess_exist   7.540557e-01 1.199797e+00


┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


In [82]:
R"""
tidied_data <- tmp_dat_ %>%
	mutate(recA = ifelse(as.integer(str_sub(line, 2, 3)) <=6, 1,0))
tidied_data %>% head(2)
"""

RObject{VecSxp}
# A tibble: 2 × 15
  first  last ess_cnt line  gen_id is_cnt ins_cnt ins_cnt_simple ess_bp
  <int> <int>   <int> <chr>  <int>  <int>   <int>          <int>  <int>
1     1 10000       0 L01-1      1      0       0              0      0
2 10001 20000       0 L01-1      1      0       0              0      0
# ℹ 6 more variables: ess_IS_bp <int>, non_ess_bp <int>, passages <dbl>,
#   ess_exist <dbl>, offset <dbl>, recA <dbl>


# Care about recA
- take recA into acount -> overdispersion in ins rate
- Use regression instead of fit -> I have not modeled  second insertion  in the offset

In [103]:
R"""
glm.ins_cnt.logit1 <- glm(Ins_tot ~ (IS + Ess) * recA,
	data = tidied_data %>% mutate(Ins_tot = ins_cnt > 0, IS= ifelse(is_cnt > 0, 1,0),
		Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

model <-  glm.ins_cnt.logit1
summary(model) %>% print
Anova(model, type = "II") %>% print
model$coefficients %>% exp %>% print
confint(model) %>% exp %>% print
""";


Call:
glm(formula = Ins_tot ~ (IS + Ess) * recA, family = binomial(link = "logit"), 
    data = tidied_data %>% mutate(Ins_tot = ins_cnt > 0, IS = ifelse(is_cnt > 
        0, 1, 0), Ess = ess_exist), offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.78876    0.08509 -185.563  < 2e-16 ***
IS            2.19250    0.15434   14.206  < 2e-16 ***
Ess          -0.08634    0.14974   -0.577  0.56422    
recA          0.84475    0.10009    8.440  < 2e-16 ***
IS:recA      -0.65066    0.19152   -3.397  0.00068 ***
Ess:recA     -0.07628    0.17783   -0.429  0.66795    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 8065  on 34983  degrees of freedom
Residual deviance: 7680  on 34978  degrees of freedom
AIC: 7692

Number of Fisher Scoring iterations: 7

Analysis of Deviance Table (Type II tests)

Response: Ins_tot
      

┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


In [104]:
R"""
glm.ins_cnt.logit1.dIS <- glm(Ins_tot ~ (Ess) * recA,
	data = tidied_data %>% mutate(Ins_tot = ins_cnt > 0, IS= ifelse(is_cnt > 0, 1,0),
		Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

glm.ins_cnt.logit1.dEss <- glm(Ins_tot ~ (IS) * recA,
	data = tidied_data %>% mutate(Ins_tot = ins_cnt > 0, IS= ifelse(is_cnt > 0, 1,0),
		Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

c(AIC.full = AIC(glm.ins_cnt.logit1), AIC.dIS = AIC(glm.ins_cnt.logit1.dIS),
	AIC.dEss = AIC(glm.ins_cnt.logit1.dEss)) %>% print
anova(glm.ins_cnt.logit1, glm.ins_cnt.logit1.dIS, test = "Chisq") %>% print
anova(glm.ins_cnt.logit1, glm.ins_cnt.logit1.dEss, test = "Chisq") %>% print
""";

AIC.full  AIC.dIS AIC.dEss 
7691.984 7966.149 7691.263 
Analysis of Deviance Table

Model 1: Ins_tot ~ (IS + Ess) * recA
Model 2: Ins_tot ~ (Ess) * recA
  Resid. Df Resid. Dev Df Deviance  Pr(>Chi)    
1     34978     7680.0                          
2     34980     7958.1 -2  -278.17 < 2.2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
Analysis of Deviance Table

Model 1: Ins_tot ~ (IS + Ess) * recA
Model 2: Ins_tot ~ (IS) * recA
  Resid. Df Resid. Dev Df Deviance Pr(>Chi)
1     34978     7680.0                     
2     34980     7683.3 -2  -3.2789   0.1941


In [102]:
R"""
glm.ins_cnt_simple.logit1 <- glm(Ins_simple ~ (IS + Ess) * recA,
	data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 0,
	IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

model <-  glm.ins_cnt_simple.logit1
summary(model) %>% print
Anova(model, type = "II") %>% print
model$coefficients %>% exp %>% print
confint(model) %>% exp %>% print
""";


Call:
glm(formula = Ins_simple ~ (IS + Ess) * recA, family = binomial(link = "logit"), 
    data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 
        0, IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist), offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -16.37242    0.11573 -141.472   <2e-16 ***
IS           -1.42640    1.00597   -1.418    0.156    
Ess          -0.08413    0.21643   -0.389    0.697    
recA          0.73316    0.13766    5.326    1e-07 ***
IS:recA      -0.02422    1.16212   -0.021    0.983    
Ess:recA      0.05053    0.25563    0.198    0.843    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 4099.0  on 34983  degrees of freedom
Residual deviance: 4040.5  on 34978  degrees of freedom
AIC: 4052.5

Number of Fisher Scoring iterations: 9

Analysis of Deviance Table (Type II tests)

Respo

┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


In [101]:
R""" # presence of IS has negative influence likely because two ISs would become complex
glm.ins_cnt_simple.logit1.dIS <- glm(Ins_simple ~ (Ess) * recA,
	data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 0,
	IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

glm.ins_cnt_simple.logit1.dEss <- glm(Ins_simple ~ (IS) * recA,
	data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 0,
	IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

c(AIC.full = AIC(glm.ins_cnt_simple.logit1), AIC.dIS = AIC(glm.ins_cnt_simple.logit1.dIS),
	AIC.dEss = AIC(glm.ins_cnt_simple.logit1.dEss)) %>% print
anova(glm.ins_cnt_simple.logit1, glm.ins_cnt_simple.logit1.dIS, test = "Chisq") %>% print
anova(glm.ins_cnt_simple.logit1, glm.ins_cnt_simple.logit1.dEss, test = "Chisq") %>% print
""";

AIC.full  AIC.dIS AIC.dEss 
4052.517 4062.292 4048.731 
Analysis of Deviance Table

Model 1: Ins_simple ~ (IS + Ess) * recA
Model 2: Ins_simple ~ (Ess) * recA
  Resid. Df Resid. Dev Df Deviance Pr(>Chi)   
1     34978     4040.5                        
2     34980     4054.3 -2  -13.775  0.00102 **
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
Analysis of Deviance Table

Model 1: Ins_simple ~ (IS + Ess) * recA
Model 2: Ins_simple ~ (IS) * recA
  Resid. Df Resid. Dev Df Deviance Pr(>Chi)
1     34978     4040.5                     
2     34980     4040.7 -2 -0.21418   0.8984


## Without interaction

In [105]:
R"""
glm.ins_cnt.logit2 <- glm(Ins_total ~ IS + Ess + recA,
	data = tidied_data %>% mutate(Ins_total = ins_cnt > 0,
	IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

model <-  glm.ins_cnt.logit2
summary(model) %>% print
Anova(model, type = "II") %>% print
model$coefficients %>% exp %>% print
confint(model) %>% exp %>% print
""";


Call:
glm(formula = Ins_total ~ IS + Ess + recA, family = binomial(link = "logit"), 
    data = tidied_data %>% mutate(Ins_total = ins_cnt > 0, IS = ifelse(is_cnt > 
        0, 1, 0), Ess = ess_exist), offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -15.68543    0.06963 -225.274   <2e-16 ***
IS            1.74781    0.09137   19.129   <2e-16 ***
Ess          -0.14320    0.08078   -1.773   0.0763 .  
recA          0.70038    0.07670    9.132   <2e-16 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 8065.0  on 34983  degrees of freedom
Residual deviance: 7691.2  on 34980  degrees of freedom
AIC: 7699.2

Number of Fisher Scoring iterations: 7

Analysis of Deviance Table (Type II tests)

Response: Ins_total
     LR Chisq Df Pr(>Chisq)    
IS    266.976  1    < 2e-16 ***
Ess     3.209  1    0.07324 .  
recA   89.

┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172


In [107]:
R"""
glm.ins_cnt_simple.logit2 <- glm(Ins_simple ~ IS + Ess + recA,
	data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 0,
	IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist),
	family = binomial(link = "logit"),
	offset = log(non_ess_bp*passages))

model <-  glm.ins_cnt_simple.logit2
summary(model) %>% print
Anova(model, type = "II") %>% print
model$coefficients %>% exp %>% print
confint(model) %>% exp %>% print
""";


Call:
glm(formula = Ins_simple ~ IS + Ess + recA, family = binomial(link = "logit"), 
    data = tidied_data %>% mutate(Ins_simple = ins_cnt_simple > 
        0, IS = ifelse(is_cnt > 0, 1, 0), Ess = ess_exist), offset = log(non_ess_bp * 
        passages))

Coefficients:
             Estimate Std. Error  z value Pr(>|z|)    
(Intercept) -16.38261    0.10323 -158.706  < 2e-16 ***
IS           -1.44468    0.50350   -2.869  0.00411 ** 
Ess          -0.04802    0.11515   -0.417  0.67666    
recA          0.74755    0.11562    6.466 1.01e-10 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1

(Dispersion parameter for binomial family taken to be 1)

    Null deviance: 4099.0  on 34983  degrees of freedom
Residual deviance: 4040.6  on 34980  degrees of freedom
AIC: 4048.6

Number of Fisher Scoring iterations: 8

Analysis of Deviance Table (Type II tests)

Response: Ins_simple
     LR Chisq Df Pr(>Chisq)    
IS     13.777  1  0.0002059 ***
Ess     0.175  1  0.6756864    


┌ Warning: RCall.jl: Waiting for profiling to be done...
└ @ RCall ~/.julia/packages/RCall/gOwEW/src/io.jl:172
